# Project 68 — Local Cost & Latency Benchmark
## Profile Model Speed × Task Type × Config Variations

**Stack:** LangChain · Ollama · pandas · Jupyter

In [ ]:
# !pip install -q langchain langchain-ollama pandas

## Step 1 — Define Benchmark Matrix

In [ ]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
import time, pandas as pd, json

configs = [
    {"name": "precise",  "temp": 0.0, "desc": "Deterministic"},
    {"name": "balanced", "temp": 0.3, "desc": "Balanced"},
    {"name": "creative", "temp": 0.8, "desc": "High creativity"},
]

tasks = [
    ("classify", "Classify as positive/negative/neutral: 'The product exceeded expectations!'"),
    ("extract", "Extract name and date: 'Meeting with Dr. Sarah Chen on March 15, 2025'"),
    ("summarize", "Summarize in 1 sentence: 'Machine learning models learn patterns from training data. "
     "They generalize to new unseen data. Overfitting occurs when models memorize training data.'"),
    ("generate", "Write a product description for noise-cancelling headphones under $100"),
    ("reason", "A snail climbs 3 feet up a wall during the day, slides 2 feet at night. "
     "How many days to reach the top of a 10-foot wall?"),
    ("code", "Write a Python function to check if a number is prime"),
]

print(f"Benchmark: {len(configs)} configs × {len(tasks)} tasks = {len(configs)*len(tasks)} runs")

## Step 2 — Execute Benchmark

In [ ]:
results = []
for config in configs:
    llm = ChatOllama(model="qwen3:8b", temperature=config["temp"])
    chain = ChatPromptTemplate.from_template("{task}") | llm | StrOutputParser()
    for task_name, prompt in tasks:
        start = time.time()
        output = chain.invoke({"task": prompt})
        elapsed = time.time() - start
        results.append({
            "config": config["name"],
            "task": task_name,
            "latency_s": round(elapsed, 2),
            "input_tokens": len(prompt.split()),
            "output_tokens": len(output.split()),
            "output_chars": len(output),
        })
        print(f"  {config['name']:<10} {task_name:<12} {elapsed:.1f}s  {len(output)} chars")

df = pd.DataFrame(results)
print(f"\nTotal runs: {len(df)}")

## Step 3 — Latency Analysis

In [ ]:
# Pivot: task × config
pivot_latency = df.pivot_table(index="task", columns="config", values="latency_s", aggfunc="mean")
print("LATENCY (seconds) — Task × Config")
print("=" * 50)
print(pivot_latency.round(2).to_string())

pivot_output = df.pivot_table(index="task", columns="config", values="output_chars", aggfunc="mean")
print("\nOUTPUT LENGTH (chars) — Task × Config")
print("=" * 50)
print(pivot_output.round(0).to_string())

## Step 4 — Throughput & Efficiency

In [ ]:
# Tokens per second estimate
df["tokens_per_sec"] = df["output_tokens"] / df["latency_s"]

print("EFFICIENCY METRICS")
print("=" * 50)
print(f"Avg latency:        {df['latency_s'].mean():.2f}s")
print(f"P50 latency:        {df['latency_s'].median():.2f}s")
print(f"P95 latency:        {df['latency_s'].quantile(0.95):.2f}s")
print(f"Avg tokens/sec:     {df['tokens_per_sec'].mean():.1f}")
print(f"Total output tokens:{df['output_tokens'].sum()}")

print("\nBy config:")
by_config = df.groupby("config").agg({
    "latency_s": ["mean", "std"],
    "output_chars": "mean",
    "tokens_per_sec": "mean",
}).round(2)
print(by_config.to_string())

print("\nBy task:")
by_task = df.groupby("task").agg({
    "latency_s": ["mean", "min", "max"],
    "output_chars": "mean",
}).round(2)
print(by_task.to_string())

## What You Learned
- **Systematic benchmarking** across configs and tasks
- **Latency profiling** with P50/P95 percentiles
- **Throughput measurement** (tokens/sec)
- **Task-specific performance** characteristics